# 04 Conditional history benchmark

Use the Python kernel. Set analysis_run_dir and raw_root in benchmark_config.json. The annual analysis directory must contain DATA_READY and data/*.parquet.

The workflow preserves focal references, outcomes, team slots, covariates, and fixed effects while reassigning complete author histories. Matching scenarios use broad fields, dominant historical sources, restricted team assignments, and narrower history-length bins.

In [ ]:
from pathlib import Path
import sys, importlib
import pandas as pd
PROJECT=Path.cwd().resolve()
assert (PROJECT/"benchmark_config.json").exists(), "Run this notebook from the repository root."
sys.path.insert(0,str(PROJECT))
import benchmark_profiles as profiles
import benchmark_randomization as randomization
importlib.reload(profiles)
importlib.reload(randomization)
MODE="new_clusters"  # new_clusters or extend_existing
assert MODE in {"new_clusters","extend_existing"}
c=profiles.config(PROJECT)
if MODE=="extend_existing":
    previous=Path(c["previous_run_dir"])
    if not previous.is_absolute():previous=PROJECT/previous
    c=profiles.existing_config(previous,PROJECT)
else:
    c=profiles.activate(c)
print("Mode:",MODE,"Run:",c["_out"],"Random worlds:",c["replicates"])

Use new_clusters to prepare the configured member-cluster sample and matching scenarios. Use extend_existing to append random worlds to a compatible completed run. Appending retains the original sample and assignment scheme.

In [ ]:
if MODE=="new_clusters":
    prepared=profiles.prepare(c)
    print("Source profiles ready:",prepared)
else:
    print("Reusing existing complete source profiles")

In [ ]:
scenario_paths=randomization.run_scenarios(c)
for path in scenario_paths:
    print(path.name)
    display(pd.read_csv(path/"matching_coverage.csv"))
    if (path/"SCENARIO_EMPTY").exists():print("No complete teams satisfy this scenario; no model will be forced.")

The default is 999 randomized worlds in addition to the observed world. Computation continues by year and world when matching settings are unchanged. Change run_name for a different sample or input dataset. Default limits are 88 GiB per process and at least 12 GiB of available system memory. Continue with notebook 05 after preparation.